# Variaveis e importacoes

In [3]:
import sqlite3
from pathlib import Path
import sys
import os
import pandas as pd
from api_anbima import fetch_data, fetch_data_for_dates
#from api_anbima.funcoes_api import fetch_data

DB_PATH = Path(r"Z:\Chila\projetos\calculadora_titulos_publicos\database\banco_principal\banco_de_dados.db")

ModuleNotFoundError: No module named 'api_anbima'

In [1]:
from titulospub import *


In [ ]:
var = var

In [2]:

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

if not tables:
    print("Banco está vazio (sem tabelas).")
else:
    vazio = True
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
        count = cursor.fetchone()[0]
        if count > 0:
            vazio = False
            print(f"Tabela {table[0]} possui {count} registros.")
    if vazio:
        print("Banco está vazio (tabelas sem registros).")

conn.close()


Tabela TITULOS_PUBLICOS possui 118 registros.
Tabela sqlite_sequence possui 1 registros.
Tabela MERCADO_SECUNDARIO possui 98566 registros.


In [3]:

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

if not tables:
    print("Banco está vazio (sem tabelas).")
else:
    vazio = True
    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
        count = cursor.fetchone()[0]
        if count > 0:
            vazio = False
            print(f"Tabela {table[0]} possui {count} registros.")
    if vazio:
        print("Banco está vazio (tabelas sem registros).")

conn.close()


Tabela TITULOS_PUBLICOS possui 118 registros.
Tabela sqlite_sequence possui 1 registros.
Tabela MERCADO_SECUNDARIO possui 98566 registros.


# Criação de Tabelas

In [26]:
def create_table(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS TITULOS_PUBLICOS (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        expressao TEXT,
        data_vencimento TEXT NOT NULL,  -- formato ISO: YYYY-MM-DD
        tipo_titulo TEXT NOT NULL,
        data_base TEXT,
        codigo_selic TEXT,
        codigo_isin TEXT,
        status TEXT NOT NULL DEFAULT 'ATIVO'
               CHECK (status IN ('ATIVO','INATIVO','SUSPENSO','CANCELADO','RESGATADO')),
        UNIQUE(tipo_titulo, data_vencimento)  -- garante unicidade do conjunto
    );
    """)
    conn.commit()
    conn.close()
    print("Tabela criada com sucesso!")

def create_table_mercado_secundario(db_path: str = DB_PATH):
    """
    Cria a tabela MERCADO_SECUNDARIO com PK composta (titulo_id, data_referencia)
    e FK para TITULOS_PUBLICOS(id). Usa WITHOUT ROWID para melhor eficiência.
    """
    ddl = """
    CREATE TABLE IF NOT EXISTS MERCADO_SECUNDARIO (
        id        INTEGER NOT NULL,              -- FK para TITULOS_PUBLICOS(id)
        data_referencia  TEXT    NOT NULL,              -- ISO YYYY-MM-DD
        taxa_anbima      REAL,
        intervalo_min_d0 REAL,
        intervalo_max_d0 REAL,
        intervalo_min_d1 REAL,
        intervalo_max_d1 REAL,
        pu               REAL,
        PRIMARY KEY (id, data_referencia),
        FOREIGN KEY (id) REFERENCES TITULOS_PUBLICOS(id)
    ) WITHOUT ROWID;
    """
    idx_data = "CREATE INDEX IF NOT EXISTS ix_mercado_data ON MERCADO_SECUNDARIO(data_referencia);"

    with sqlite3.connect(db_path, timeout=10) as conn:
        cur = conn.cursor()
        cur.execute("PRAGMA foreign_keys = ON;")
        cur.execute(ddl)
        cur.execute(idx_data)
        conn.commit()
    print("Tabela MERCADO_SECUNDARIO criada com PK composta e índice em data_referencia.")



In [27]:
create_table_mercado_secundario()

Tabela MERCADO_SECUNDARIO criada com PK composta e índice em data_referencia.


# Scraping

## Tabela: MERCADO_SECUNDARIO

## gerando uma lista de datas faltantes manuelmente

In [44]:
#Gerando lista com datas

from datetime import datetime, timedelta

data_inicial = "2017-01-01"
data_final = "2026-01-16"

# Converte para objetos datetime
start_date = datetime.strptime(data_inicial, "%Y-%m-%d")
end_date = datetime.strptime(data_final, "%Y-%m-%d")

# Se as datas inicial e final forem iguais, retorna lista com uma data só
if start_date == end_date:
    lista_datas = [start_date.strftime("%Y-%m-%d")]
else:
    # Gera lista de datas (inclusive inicial e final)
    lista_datas = []
    current_date = start_date
    while current_date <= end_date:
        lista_datas.append(current_date.strftime("%Y-%m-%d"))
        current_date += timedelta(days=1)

print(lista_datas)

['2017-01-01', '2017-01-02', '2017-01-03', '2017-01-04', '2017-01-05', '2017-01-06', '2017-01-07', '2017-01-08', '2017-01-09', '2017-01-10', '2017-01-11', '2017-01-12', '2017-01-13', '2017-01-14', '2017-01-15', '2017-01-16', '2017-01-17', '2017-01-18', '2017-01-19', '2017-01-20', '2017-01-21', '2017-01-22', '2017-01-23', '2017-01-24', '2017-01-25', '2017-01-26', '2017-01-27', '2017-01-28', '2017-01-29', '2017-01-30', '2017-01-31', '2017-02-01', '2017-02-02', '2017-02-03', '2017-02-04', '2017-02-05', '2017-02-06', '2017-02-07', '2017-02-08', '2017-02-09', '2017-02-10', '2017-02-11', '2017-02-12', '2017-02-13', '2017-02-14', '2017-02-15', '2017-02-16', '2017-02-17', '2017-02-18', '2017-02-19', '2017-02-20', '2017-02-21', '2017-02-22', '2017-02-23', '2017-02-24', '2017-02-25', '2017-02-26', '2017-02-27', '2017-02-28', '2017-03-01', '2017-03-02', '2017-03-03', '2017-03-04', '2017-03-05', '2017-03-06', '2017-03-07', '2017-03-08', '2017-03-09', '2017-03-10', '2017-03-11', '2017-03-12', '2017

## Consultando datas faltantes na tabela

In [1]:
from database.utils.auxilio import obter_datas_faltantes

datas_faltantes = obter_datas_faltantes("MERCADO_SECUNDARIO", "data_referencia")[1]
datas_faltantes

['2026-01-20', '2026-01-21']

In [7]:
from clients.anbima import AnbimaClient

ModuleNotFoundError: No module named 'clients'

In [6]:
#Buscando os dados na API da anbima
#Buscando os dados na API da anbima
url_mercado_secundario = "https://api.anbima.com.br/feed/precos-indices/v1/titulos-publicos/mercado-secundario-TPF"
dados = fetch_data_for_dates(url_mercado_secundario, datas_faltantes)
dados

NameError: name 'fetch_data_for_dates' is not defined

In [4]:
#transformando em df

import pandas as pd

# Supondo que 'd' seja: List[List[dict]]
# Ex.: d = [ [ {dict do dia X}, {dict do dia X} ], [ {dict do dia Y}, ... ], ... ]

# 1) Achatar a lista de listas em uma lista simples de dicts
registros = [item for lista_do_dia in dados for item in (lista_do_dia or []) if isinstance(item, dict)]

# 2) Criar o DataFrame
df = pd.DataFrame.from_records(registros)


# 4) Ordenar (opcional)
cols_ordem = [c for c in ["data_referencia", "tipo_titulo"] if c in df.columns]
if cols_ordem:
    df = df.sort_values(cols_ordem, kind="stable").reset_index(drop=True)

df = df.rename(columns={"taxa_indicativa": "taxa_anbima"})
df


,tipo_titulo,expressao,data_vencimento,data_referencia,codigo_selic,data_base,taxa_compra,taxa_venda,taxa_anbima,intervalo_min_d0,intervalo_max_d0,intervalo_min_d1,intervalo_max_d1,pu,desvio_padrao,codigo_isin
0,LFT,Rentabilidade (% a.a.)/252,2026-03-01,2026-01-16,210100,2000-07-01,0.0251,0.0195,0.0218,-0.0479,0.0673,-0.0495,0.0673,18195.261471,0.001471,BRSTNCLF1RE0
1,LFT,Rentabilidade (% a.a.)/252,2026-09-01,2026-01-16,210100,2000-07-01,-0.0145,-0.0197,-0.0159,-0.0400,0.0563,-0.0415,0.0571,18197.517742,0.001565,BRSTNCLF1RF7
2,LFT,Rentabilidade (% a.a.)/252,2027-03-01,2026-01-16,210100,2000-07-01,0.0233,0.0200,0.0219,-0.0137,0.0393,-0.0142,0.0394,18191.349388,0.000115,BRSTNCLF1RG5
3,LFT,Rentabilidade (% a.a.)/252,2027-09-01,2026-01-16,210100,2000-07-01,0.0330,0.0310,0.0320,0.0007,0.0502,0.0001,0.0503,18186.345561,0.000000,BRSTNCLF1RH3
4,LFT,Rentabilidade (% a.a.)/252,2028-03-01,2026-01-16,210100,2000-07-01,0.0455,0.0437,0.0446,0.0256,0.0540,0.0247,0.0543,18178.666961,0.000482,BRSTNCLF1RI1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,NTN-F,Taxa (% a.a.)/252,2029-01-01,2026-01-19,950199,2018-01-05,13.2887,13.2587,13.2768,12.8402,13.8640,12.7955,13.8192,933.152884,0.003914,BRSTNCNTF1Q6
100,NTN-F,Taxa (% a.a.)/252,2031-01-01,2026-01-19,950199,2020-01-10,13.6788,13.6683,13.6740,13.2605,14.2410,13.2338,14.2143,884.966297,0.000000,BRSTNCNTF204
101,NTN-F,Taxa (% a.a.)/252,2033-01-01,2026-01-19,950199,2022-01-07,13.8552,13.8427,13.8475,13.4186,14.3928,13.4025,14.3765,846.971192,0.002850,BRSTNCNTF212
102,NTN-F,Taxa (% a.a.)/252,2035-01-01,2026-01-19,950199,2024-01-05,13.8735,13.8622,13.8680,13.4438,14.4089,13.4254,14.3903,821.719889,0.000000,BRSTNCNTF238


## Tabela: BMF

In [46]:
import time

def definir_caminho_adj_bmf(data_str):
    """
    Versão modificada que busca o arquivo mais recente por data de modificação,
    não apenas pelo número no nome do arquivo.
    
    Args:
        data_str: String no formato "AAAA-MM-DD" (ex: "2026-08-15")
    
    Returns:
        Caminho completo do arquivo mais recente ou None se não encontrado
    """
    try:
        # Converter string para datetime
        from datetime import datetime
        data = datetime.strptime(data_str, "%Y-%m-%d")
        
        # Convertendo para string formatada
        dia = f"{data.day:02}"
        mes = f"{data.month:02}"
        ano = str(data.year)

        # Pasta
        pasta = f'x:\\Interest_Rate\\SettlementPrice\\{ano}{mes}{dia}\\'

        # Prefixo fixo do arquivo
        prefixo = f'Interest_Rate_SettlementPriceFile_Futures_{ano}{mes}{dia}_'

        # Verifica se a pasta existe
        if not os.path.exists(pasta):
            print(f"[AVISO] Pasta não encontrada: {pasta}")
            return None

        # Lista todos arquivos
        arquivos = os.listdir(pasta)

        # Filtra os que começam com prefixo e terminam com .csv
        arquivos_filtrados = [
            f for f in arquivos
            if f.startswith(prefixo) and f.endswith(".csv")
        ]

        if not arquivos_filtrados:
            print(f"[AVISO] Nenhum arquivo encontrado com prefixo {prefixo} na pasta {pasta}")
            return None

        # Busca o arquivo mais recente por data de modificação
        arquivos_com_data = []
        for arq in arquivos_filtrados:
            caminho_completo = os.path.join(pasta, arq)
            if os.path.exists(caminho_completo):
                mtime = os.path.getmtime(caminho_completo)
                arquivos_com_data.append((mtime, arq, caminho_completo))

        if not arquivos_com_data:
            print(f"[AVISO] Nenhum arquivo válido encontrado na pasta {pasta}")
            return None

        # Pega o arquivo com maior data de modificação (mais recente)
        arquivo_mais_recente = max(arquivos_com_data, key=lambda x: x[0])
        
        print(f"[INFO] Encontrados {len(arquivos_filtrados)} arquivos com prefixo {prefixo}")
        print(f"[INFO] Arquivo mais recente selecionado: {arquivo_mais_recente[1]}")
        print(f"[INFO] Data de modificação: {time.ctime(arquivo_mais_recente[0])}")

        return arquivo_mais_recente[2]  # Retorna caminho completo
    
    except Exception as e:
        print(f"[AVISO] Erro inesperado ao definir caminho: {e}")
        import traceback
        traceback.print_exc()
        return None

def scrap_ajustes_bmf(data):

    caminho = definir_caminho_adj_bmf(data)

    return pd.read_csv(caminho, sep=";")

def scrap_ajustes_bmf_multiplas_datas(lista_datas):
    """
    Processa múltiplas datas e retorna um único DataFrame com todos os dados.
    
    Args:
        lista_datas: Lista de strings no formato "AAAA-MM-DD" (ex: ["2026-08-15", "2026-08-16"])
    
    Returns:
        pandas.DataFrame com todos os dados concatenados. DataFrame vazio se nenhum arquivo for encontrado.
    """
    dfs = []
    
    for data_str in lista_datas:
        try:
            print(f"\n[INFO] Processando data: {data_str}")
            
            # Busca o caminho do arquivo
            caminho = definir_caminho_adj_bmf(data_str)
            
            if caminho is None:
                print(f"[AVISO] Pulando data {data_str} - arquivo não encontrado")
                continue
            
            # Lê o CSV
            df_temp = pd.read_csv(caminho, sep=";")
            
            # Adiciona coluna com a data de referência (opcional, mas útil)
            if 'data_referencia' not in df_temp.columns:
                df_temp['data_referencia'] = data_str
            
            dfs.append(df_temp)
            print(f"[INFO] Data {data_str} processada com sucesso. {len(df_temp)} registros.")
            
        except Exception as e:
            print(f"[ERRO] Erro ao processar data {data_str}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Concatena todos os DataFrames
    if dfs:
        df_final = pd.concat(dfs, ignore_index=True)
        print(f"\n[INFO] Total de registros no DataFrame final: {len(df_final)}")
        return df_final
    else:
        print("[AVISO] Nenhum arquivo foi encontrado/processado. Retornando DataFrame vazio.")
        return pd.DataFrame()

In [61]:


df_bmf  = scrap_ajustes_bmf_multiplas_datas(lista_datas)


df_bmf.rename(columns={"RptDt": "data_referencia", 
                       "TckrSymb": "ticker", 
                       "ISIN": "codigo_isin", 
                       "XprtnDt": "data_vencimento", 
                       "AdjstdQtTax": "ajuste",
                       "AdjstdQt": "ajuste_quantidade"}, inplace=True)

df_bmf = df_bmf[df_bmf['ticker'].str.startswith(('DAP', 'DI1'))]


[INFO] Processando data: 2017-01-01
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170101\
[AVISO] Pulando data 2017-01-01 - arquivo não encontrado

[INFO] Processando data: 2017-01-02
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170102\
[AVISO] Pulando data 2017-01-02 - arquivo não encontrado

[INFO] Processando data: 2017-01-03
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170103\
[AVISO] Pulando data 2017-01-03 - arquivo não encontrado

[INFO] Processando data: 2017-01-04
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170104\
[AVISO] Pulando data 2017-01-04 - arquivo não encontrado

[INFO] Processando data: 2017-01-05
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170105\
[AVISO] Pulando data 2017-01-05 - arquivo não encontrado

[INFO] Processando data: 2017-01-06
[AVISO] Pasta não encontrada: x:\Interest_Rate\SettlementPrice\20170106\
[AVISO] Pulando data 2017-01-06 - arquivo não encontrad

In [62]:


df_bmf_1 = df_bmf.drop_duplicates(subset=["ticker"])
df_bmf_1







,data_referencia,ticker,SctyId,SctySrc,MktIdrCd,codigo_isin,data_vencimento,ajuste_quantidade,ajuste,AdjstdQtStin,PrvsAdjstdQt,PrvsAdjstdQtTax,PrvsAdjstdQtStin,VartnPts,EqvtVal,AdjstdValCtrct,DataSts,data_referencia
0,2023-04-14,DAPF24,200000760613,8,BVMF,BRBMEFDAP3S3,2024-01-15,94938.70,7.250,F,95002.76,7.15,U,-64.06,NaN,-105.819753,I,2023-04-14
1,2023-04-14,DAPF25,100000191317,8,BVMF,BRBMEFDAP470,2025-01-15,90574.60,5.820,F,90691.41,5.74,U,-116.81,NaN,-192.956687,I,2023-04-14
2,2023-04-14,DAPJ23,100000191360,8,BVMF,BRBMEFDAP488,2023-04-17,99981.40,4.800,F,99981.47,4.78,U,-0.07,NaN,-0.115632,I,2023-04-14
3,2023-04-14,DAPK23,388396,8,BVMF,BRBMEFDAP1G2,2023-05-15,99519.27,6.600,F,99507.43,6.77,U,11.84,NaN,19.558318,I,2023-04-14
4,2023-04-14,DAPK25,200000356461,8,BVMF,BRBMEFDAP330,2025-05-15,89169.33,5.690,F,89271.55,5.63,U,-102.22,NaN,-168.855685,I,2023-04-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122461,2026-01-02,DI1V31,400000113862,8,BVMF,BRBMEFD1I8S6,2031-10-01,48837.15,13.392,F,NaN,NaN,NaN,NaN,NaN,NaN,N,2026-01-02
123091,2026-01-07,DI1F41,300000055524,8,BVMF,BRBMEFD1I8T4,2041-01-02,15122.89,13.527,F,NaN,NaN,NaN,NaN,NaN,NaN,N,2026-01-07
124307,2026-01-15,DAPK31,300000055944,8,BVMF,BRBMEFDAP5E8,2031-05-15,67204.59,7.821,F,NaN,NaN,NaN,NaN,NaN,NaN,I,2026-01-15
124310,2026-01-15,DAPK37,300000055945,8,BVMF,BRBMEFDAP5F5,2037-05-15,44274.62,7.511,F,NaN,NaN,NaN,NaN,NaN,NaN,I,2026-01-15


# Inserindo dados

## tabela: TITULOS_PUBLICOS

In [15]:
# Conexão com SQLite
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Inserindo linha por linha com mapeamento explícito
for _, row in df_1.iterrows():
    cursor.execute("""
        INSERT INTO TITULOS_PUBLICOS (expressao, data_vencimento, tipo_titulo, data_base, codigo_selic, codigo_isin)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row['expressao'], row['data_vencimento'], row['tipo_titulo'], row['data_base'], row['codigo_selic'], row['codigo_isin']))

conn.commit()
conn.close()

In [9]:
df_1 = df[df["data_vencimento"].isin(["2031-05-15", "2037-05-15"])]
df_1

,tipo_titulo,expressao,data_vencimento,data_referencia,codigo_selic,data_base,taxa_compra,taxa_venda,taxa_anbima,intervalo_min_d0,intervalo_max_d0,intervalo_min_d1,intervalo_max_d1,pu,desvio_padrao,codigo_isin
135,NTN-B,Taxa (% a.a.)/252,2031-05-15,2026-01-14,760199,2000-07-15,7.8857,7.8546,7.8699,NaN,NaN,7.5785,8.0819,4286.304780,0.000289,BRSTNCNTB7X3
139,NTN-B,Taxa (% a.a.)/252,2037-05-15,2026-01-14,760199,2000-07-15,7.6291,7.5751,7.5989,NaN,NaN,7.4128,7.8039,4109.991798,0.012117,BRSTNCNTB7Y1


## Tabela: MERCADO_SECUNDARIO

In [5]:

def inserir_mercado_secundario(df, db_path=DB_PATH):
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()

    for _, row in df.iterrows():
        tipo_titulo = row["tipo_titulo"]
        data_vencimento = row["data_vencimento"]

        # Buscar o ID na tabela TITULOS_PUBLICOS
        cursor.execute("""
            SELECT id FROM TITULOS_PUBLICOS
            WHERE tipo_titulo = ? AND data_vencimento = ?
        """, (tipo_titulo, data_vencimento))
        resultado = cursor.fetchone()

        if resultado:
            fk_id = resultado[0]

            # Inserir na tabela MERCADO_SECUNDARIO
            cursor.execute("""
                INSERT INTO MERCADO_SECUNDARIO (
                    id, data_referencia, taxa_anbima,
                    intervalo_min_d0, intervalo_max_d0,
                    intervalo_min_d1, intervalo_max_d1, pu
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                fk_id,
                str(row["data_referencia"]),
                row["taxa_anbima"],
                row["intervalo_min_d0"],
                row["intervalo_max_d0"],
                row["intervalo_min_d1"],
                row["intervalo_max_d1"],
                row["pu"]
            ))
        else:
            print(f"⚠️ Não encontrado TITULOS_PUBLICOS para {tipo_titulo} - {data_vencimento}")

    conn.commit()
    conn.close()
    print("Inserção concluída.")

# Exemplo de uso:
 #df_1 é seu DataFrame já carregado e com colunas corretas
inserir_mercado_secundario(df)


Inserção concluída.


# Consultas

## Consultando anbimas

In [4]:

import sqlite3
import pandas as pd

def buscar_mercado_secundario(
    db_path,
    data_ini: str,
    data_fim: str,
    tipo_titulo: list[str] | None = None,
    data_vencimento: list[str] | None = None,
    status: list[str] | None = None,
    columns_ms: list[str] | None = None,
) -> pd.DataFrame:
    """
    Retorna um DataFrame com as colunas da tabela MERCADO_SECUNDARIO
    juntamente com tipo_titulo e data_vencimento da tabela TITULOS_PUBLICOS,
    filtrando por um intervalo [data_ini, data_fim] em data_referencia.

    Parâmetros:
        db_path         : caminho do arquivo SQLite.
        data_ini        : string 'YYYY-MM-DD' (inclusive).
        data_fim        : string 'YYYY-MM-DD' (inclusive).
        tipo_titulo     : lista opcional de tipos para filtrar (e.g., ["NTNB", "LTN"]).
        data_vencimento : lista opcional de datas de vencimento 'YYYY-MM-DD' para filtrar.
        status          : lista opcional de status do título (e.g., ["ATIVO"]).
        columns_ms      : lista opcional de colunas específicas de MERCADO_SECUNDARIO
                          para retornar. Se None, retorna todas.

    Retorna:
        pandas.DataFrame com colunas:
            tipo_titulo, data_vencimento, data_referencia, (colunas de MS...)
    """
    # Normalização básica das datas de entrada (garante formato ISO)
    data_ini = pd.to_datetime(data_ini).strftime("%Y-%m-%d")
    data_fim = pd.to_datetime(data_fim).strftime("%Y-%m-%d")

    # Colunas padrão de MS (se não for especificado, traz todas)
    all_ms_cols = [
        "id",
        "data_referencia",
        "taxa_anbima",
        "intervalo_min_d0",
        "intervalo_max_d0",
        "intervalo_min_d1",
        "intervalo_max_d1",
        "pu",
    ]
    if columns_ms is None:
        select_ms_cols = ", ".join([f"ms.{c}" for c in all_ms_cols])
    else:
        # valida as colunas solicitadas
        invalid = [c for c in columns_ms if c not in all_ms_cols]
        if invalid:
            raise ValueError(f"Colunas inválidas em MERCADO_SECUNDARIO: {invalid}")
        select_ms_cols = ", ".join([f"ms.{c}" for c in columns_ms])

    # Base do SELECT
    select_cols = f"""
        tp.tipo_titulo,
        tp.data_vencimento,
        {select_ms_cols}
    """

    sql = f"""
        SELECT
            {select_cols}
        FROM MERCADO_SECUNDARIO ms
        JOIN TITULOS_PUBLICOS tp
          ON tp.id = ms.id
        WHERE ms.data_referencia BETWEEN ? AND ?
    """

    params: list = [data_ini, data_fim]

    # Filtros opcionais
    if tipo_titulo:
        placeholders = ", ".join(["?"] * len(tipo_titulo))
        sql += f" AND tp.tipo_titulo IN ({placeholders})"
        params.extend(tipo_titulo)

    if data_vencimento:
        # Garante ISO para o filtro
        data_vencimento_iso = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in data_vencimento]
        placeholders = ", ".join(["?"] * len(data_vencimento_iso))
        sql += f" AND tp.data_vencimento IN ({placeholders})"
        params.extend(data_vencimento_iso)

    if status:
        placeholders = ", ".join(["?"] * len(status))
        sql += f" AND tp.status IN ({placeholders})"
        params.extend(status)

    # Ordenação sugerida
    sql += """
        ORDER BY tp.tipo_titulo, tp.data_vencimento, ms.data_referencia
    """

    # Conexão com ajustes para reduzir "database is locked"
    with sqlite3.connect(db_path, timeout=30) as conn:
        conn.execute("PRAGMA foreign_keys = ON;")
        conn.execute("PRAGMA busy_timeout = 30000;")
        # WAL acelera leituras concorrentes; se você já estiver usando, tudo bem
        conn.execute("PRAGMA journal_mode = WAL;")
        conn.execute("PRAGMA synchronous = NORMAL;")

        df = pd.read_sql_query(sql, conn, params=params)

    # Tipagem opcional: converter datas para datetime e ordenar
    if not df.empty:
        for col in ["data_vencimento", "data_referencia"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.sort_values(["tipo_titulo", "data_vencimento", "data_referencia"]).reset_index(drop=True)

    return df


In [5]:
dados = buscar_mercado_secundario(
    db_path= DB_PATH,
    data_ini = "2020-01-01",
    data_fim = "2026-01-15"
)
dados

,tipo_titulo,data_vencimento,id,data_referencia,taxa_anbima,intervalo_min_d0,intervalo_max_d0,intervalo_min_d1,intervalo_max_d1,pu
0,LFT,2020-03-01,5,2020-01-02,0.0023,0.0001,0.0051,0.0001,0.0051,10471.849015
1,LFT,2020-03-01,5,2020-01-03,0.0023,0.0001,0.0051,0.0002,0.0050,10473.638550
2,LFT,2020-03-01,5,2020-01-06,0.0025,0.0002,0.0050,0.0004,0.0053,10475.428390
3,LFT,2020-03-01,5,2020-01-07,0.0026,0.0004,0.0053,0.0005,0.0053,10477.218535
4,LFT,2020-03-01,5,2020-01-08,0.0025,0.0005,0.0053,0.0004,0.0052,10479.008987
...,...,...,...,...,...,...,...,...,...,...
69041,NTN-F,2037-01-01,116,2026-01-09,13.7385,NaN,NaN,13.2983,14.2595,805.793046
69042,NTN-F,2037-01-01,116,2026-01-12,13.6862,13.2983,14.2595,13.2460,14.2073,808.586350
69043,NTN-F,2037-01-01,116,2026-01-13,13.7512,13.2460,14.2073,13.3115,14.2715,806.040089
69044,NTN-F,2037-01-01,116,2026-01-14,13.7789,13.3115,14.2715,13.3390,14.2995,805.196914


In [6]:
dados = dados[["tipo_titulo", "data_vencimento", "data_referencia", "taxa_anbima"]]
dados = dados[dados["tipo_titulo"] == "NTN-B"]
dados.reset_index(inplace=True, drop=True)
dados

,tipo_titulo,data_vencimento,data_referencia,taxa_anbima
0,NTN-B,2020-08-15,2020-01-02,0.1222
1,NTN-B,2020-08-15,2020-01-03,0.2300
2,NTN-B,2020-08-15,2020-01-06,0.2546
3,NTN-B,2020-08-15,2020-01-07,0.4391
4,NTN-B,2020-08-15,2020-01-08,0.5600
...,...,...,...,...
21730,NTN-B,2060-08-15,2026-01-09,7.2400
21731,NTN-B,2060-08-15,2026-01-12,7.2477
21732,NTN-B,2060-08-15,2026-01-13,7.2608
21733,NTN-B,2060-08-15,2026-01-14,7.3332


In [13]:
dados = dados[dados["data_vencimento"] >= pd.to_datetime("2026-08-15")]
dados

,tipo_titulo,data_vencimento,data_referencia,taxa_anbima
5300,NTN-B,2026-08-15,2020-01-02,2.6700
5301,NTN-B,2026-08-15,2020-01-03,2.7000
5302,NTN-B,2026-08-15,2020-01-06,2.7500
5303,NTN-B,2026-08-15,2020-01-07,2.7400
5304,NTN-B,2026-08-15,2020-01-08,2.7248
...,...,...,...,...
21730,NTN-B,2060-08-15,2026-01-09,7.2400
21731,NTN-B,2060-08-15,2026-01-12,7.2477
21732,NTN-B,2060-08-15,2026-01-13,7.2608
21733,NTN-B,2060-08-15,2026-01-14,7.3332


# Analises

## 1. Análises de Curto Prazo (Trading / Tático)

In [14]:
df = dados
df.dtypes

tipo_titulo                object
data_vencimento    datetime64[ns]
data_referencia    datetime64[ns]
taxa_anbima               float64
dtype: object

In [15]:

import pandas as pd
import plotly.express as px

df['data_referencia'] = pd.to_datetime(df['data_referencia'])
df = df.sort_values(['data_vencimento', 'data_referencia'])
df['daily_change'] = df.groupby('data_vencimento')['taxa_anbima'].diff()

ultimo_dia = df['data_referencia'].max()
df_ult_dia = df[df['data_referencia'] == ultimo_dia]

fig = px.bar(
    df_ult_dia,
    x='data_vencimento',
    y='daily_change',
    title='Variação Diária por Vencimento (bps)',
    labels={'data_vencimento': 'Vencimento', 'daily_change': 'Variação'}
)

fig.update_layout(xaxis_tickangle=45)
fig.show()



C:\Users\rborges\AppData\Local\Temp\ipykernel_13748\1959845760.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [16]:

import plotly.express as px

df['ret'] = df.groupby('data_vencimento')['taxa_anbima'].pct_change()
df['vol_30d'] = df.groupby('data_vencimento')['ret'].transform(lambda x: x.rolling(30).std())

vol_atual = (
    df.groupby('data_vencimento')['vol_30d']
      .last()
      .reset_index()
      .dropna()
)

fig = px.line(
    vol_atual,
    x='data_vencimento',
    y='vol_30d',
    title='Volatilidade 30 dias por Vencimento',
    markers=True
)

fig.update_layout(xaxis_tickangle=45)
fig.show()


In [17]:

import plotly.express as px

pivot = df.pivot(index='data_referencia', columns='data_vencimento', values='taxa_anbima')
corr = pivot.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale='RdBu_r',
    title='Correlação Entre Vértices da Curva NTNB (-1 a 1)'
)

fig.update_layout(width=900, height=900)
fig.show()


In [18]:

import numpy as np
import plotly.express as px

df['mm20'] = df.groupby('data_vencimento')['taxa_anbima'].transform(lambda x: x.rolling(20).mean())
df['zscore'] = df.groupby('data_vencimento')['taxa_anbima'].transform(
    lambda x: (x - x.rolling(20).mean()) / x.rolling(20).std()
)

z_atual = df.groupby('data_vencimento')['zscore'].last().reset_index()

fig = px.bar(
    z_atual,
    x='data_vencimento',
    y='zscore',
    title='Z-score da Taxa por Vencimento (Momentum / Overstretch)',
    labels={'data_vencimento': 'Vencimento', 'zscore': 'Z-score'}
)

fig.update_layout(
    xaxis_tickangle=45,
    shapes=[
        dict(type='line', y0=2, y1=2, x0=-0.5, x1=len(z_atual)-0.5, line=dict(color='red', dash='dash')),
        dict(type='line', y0=-2, y1=-2, x0=-0.5, x1=len(z_atual)-0.5, line=dict(color='red', dash='dash'))
    ]
)

fig.show()


In [28]:

import plotly.express as px

df['data_referencia'] = pd.to_datetime(df['data_referencia'])

fig = px.line(
    df,
    x='data_referencia',
    y='taxa_anbima',
    color='data_vencimento',
    title='Evolução Histórica das Taxas NTNB por Vencimento'
)

fig.update_layout(
    xaxis_title="Data",
    yaxis_title="Taxa ANBIMA (%)",
    legend_title="Vencimento",
    height=600
)

fig.show()


In [25]:

import plotly.graph_objects as go

pivot = df.pivot(index='data_referencia', columns='data_vencimento', values='taxa_anbima')

# Ajuste os vencimentos conforme disponíveis no seu dataset
v1 = '2027-05-15'
v2 = '2026-08-15'
v3 = '2050-08-15'
v4 = '2060-08-15'
v5 = '2040-08-15'

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pivot.index, y=pivot[v1] - pivot[v2],
    mode='lines', name=f"Spread {v1} - {v2}"
))
fig.add_trace(go.Scatter(
    x=pivot.index, y=pivot[v3] - pivot[v1],
    mode='lines', name=f"Spread {v3} - {v1}"
))
fig.add_trace(go.Scatter(
    x=pivot.index, y=pivot[v4] - pivot[v5],
    mode='lines', name=f"Spread {v4} - {v5}"
))

fig.update_layout(
    title="Inclinação da Curva (Spreads ao Longo do Tempo)",
    xaxis_title="Data",
    yaxis_title="Spread (bps)",
    height=600
)

fig.show()


In [27]:

from sklearn.decomposition import PCA
import plotly.express as px
import numpy as np

pivot = df.pivot(index='data_referencia', columns='data_vencimento', values='taxa_anbima').dropna()
pca = PCA(n_components=3)
components = pca.fit_transform(pivot.values)

df_pca = pd.DataFrame({
    'Data': pivot.index,
    'Fator 1 (Nível)': components[:, 0],
    'Fator 2 (Inclinação)': components[:, 1],
    'Fator 3 (Curvatura)': components[:, 2],
})

fig = px.line(
    df_pca,
    x='Data',
    y=['Fator 1 (Nível)', 'Fator 2 (Inclinação)', 'Fator 3 (Curvatura)'],
    title='PCA da Curva de Juros Real (NTNB)'
)

fig.update_layout(height=600)
fig.show()

print("Variância explicada por fator:", pca.explained_variance_ratio_)


: 

In [29]:

import plotly.express as px

def calc_percentil(v):
    return v.rank(pct=True)

df['percentil'] = df.groupby('data_vencimento')['taxa_anbima'].transform(calc_percentil)

percentil_atual = (
    df.groupby('data_vencimento')['percentil']
      .last()
      .reset_index()
)

fig = px.bar(
    percentil_atual,
    x='data_vencimento',
    y='percentil',
    title='Percentil Histórico das Taxas (Valuation Relativo)',
    labels={'percentil': 'Percentil'},
    range_y=[0,1]
)

fig.update_layout(xaxis_tickangle=45)
fig.show()


In [30]:

import plotly.express as px
import numpy as np

df['ret'] = df.groupby('data_vencimento')['taxa_anbima'].pct_change()

vol_daily = (
    df.groupby(['data_referencia', 'data_vencimento'])['ret']
      .std()
      .reset_index()
)

fig = px.line(
    vol_daily,
    x='data_referencia',
    y='ret',
    color='data_vencimento',
    title='Regimes de Volatilidade das NTNB por Vencimento',
    labels={'ret':'Volatilidade'}
)

fig.update_layout(height=600)
fig.show()
